# Test Example

```python
import pandas as pd

train_df = pd.read_csv('train.csv')
test_df = pd.read_csv('test.csv')
submission = pd.read_csv('sample_submission.csv')
train_df.head(2)
```

## 전처리 

```python
import numpy as np
from sklearn.preprocessing import LabelEncoder

# 파생 변수 생성 1 : 날짜, 시간정보 생성
time_pattern = r'(\d{4})-(\d{1,2})-(\d{1,2}) (\d{1,2})' 
train_df[['연', '월', '일', '시간']] = train_df['사고일시'].str.extract(time_pattern)
train_df[['연', '월', '일', '시간']] = train_df[['연', '월', '일', '시간']].apply(pd.to_numeric) # 추출된 문자열을 수치화해줍니다 
test_df[['연', '월', '일', '시간']] = test_df['사고일시'].str.extract(time_pattern)
test_df[['연', '월', '일', '시간']] = test_df[['연', '월', '일', '시간']].apply(pd.to_numeric)
train_df = train_df.drop(columns=['사고일시']) # 정보 추출이 완료된 '사고일시' 컬럼은 제거합니다 
test_df = test_df.drop(columns=['사고일시'])


# 파생 변수 생성 2 : 공간(위치) 정보 생성
location_pattern = r'(\S+) (\S+) (\S+)'
train_df[['도시', '구', '동']] = train_df['시군구'].str.extract(location_pattern)
test_df[['도시', '구', '동']] = test_df['시군구'].str.extract(location_pattern)
train_df = train_df.drop(columns=['시군구'])
test_df = test_df.drop(columns=['시군구'])


# 파생 변수 추출 3 : 도로 형태 정보 추출
road_pattern = r'(.+) - (.+)'
train_df[['도로형태1', '도로형태2']] = train_df['도로형태'].str.extract(road_pattern)
test_df[['도로형태1', '도로형태2']] = test_df['도로형태'].str.extract(road_pattern)
train_df = train_df.drop(columns=['도로형태'])
test_df = test_df.drop(columns=['도로형태'])


# train, test 데이터의 독립변수, 종속 변수 설정
test = test_df.drop(columns=['ID']).copy()
x_train = train_df[test.columns].copy()
y_train = train_df['ECLO'].copy()


# 범주형(Categorical) 변수, 수치형 변수로 변환
categorical_features = list(x_train.dtypes[x_train.dtypes == "object"].index)
for i in categorical_features:
    le = LabelEncoder()
    le=le.fit(x_train[i]) 
    x_train[i]=le.transform(x_train[i])
    
    for case in np.unique(test[i]):
        if case not in le.classes_: 
            le.classes_ = np.append(le.classes_, case) 
    test[i]=le.transform(test[i])
```

## 가중 펴균 앙상블

### 모델 학습 

```python
from xgboost import XGBRegressor
from lightgbm import LGBMRegressor
from sklearn.ensemble import GradientBoostingRegressor

# 모델 학습
model_xgb = XGBRegressor(n_estimators=20, random_state=42)
model_gra = GradientBoostingRegressor(n_estimators=20, random_state=42)
model_lgb = LGBMRegressor(n_estimators=20, random_state=42)

model_xgb.fit(x_train, y_train)
model_gra.fit(x_train, y_train)
model_lgb.fit(x_train, y_train)

# 예측
pred_xgb = model_xgb.predict(test)
pred_gra = model_gra.predict(test)
pred_lgb = model_lgb.predict(test)
```

### 가중 평균 앙상블 적용

```python
final_pred = 0.1 * pred_xgb + 0.4 * pred_gra + 0.5 * pred_lgb

weighted_averaging_submission = submission.copy()
weighted_averaging_submission['ECLO'] = final_pred
weighted_averaging_submission
```

### 평균 앙상블 및 가중 평균 앙상블 구현

```python
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, MinMaxScaler
from lightgbm import LGBMRegressor

def train_model(x, y, split_random_state, num_estimators, scaler=None):
    # 데이터 분할
    X_train, X_test, y_train, y_test = train_test_split(x, y, test_size=0.15, random_state=split_random_state)
    
    # 스케일링 적용
    if scaler:
        scaler = scaler()
        X_train = scaler.fit_transform(X_train)
        X_test = scaler.transform(X_test)
    
    # 모델 학습
    model = LGBMRegressor(n_estimators=num_estimators, random_state=288)
    model.fit(X_train, y_train)
    
    return model


case_1 = train_model(x_train, y_train, 14414, 30)
case_2 = train_model(x_train, y_train, 998, 40)
case_3 = train_model(x_train, y_train, 22882, 50)
case_4 = train_model(x_train, y_train, 3402, 60)
case_5 = train_model(x_train, y_train, 14414, 70, scaler=StandardScaler)
case_6 = train_model(x_train, y_train, 14414, 80, scaler=MinMaxScaler)

result_1_list = [case_1, case_2, case_3, case_4]
result_2_list = [case_5, case_6]

result1, result2 = 0, 0

for model_1 in result_1_list:
    result1 += model_1.predict(test) / 4
    
for model_2 in result_2_list:
    result2 += model_2.predict(test) / 2
    
final_result = (0.7 * result1) + (0.3 * result2)

averaging_submission = submission.copy()
averaging_submission['ECLO'] = final_result
averaging_submission.head()
```

### 


## 시드 기반 KFold 앙상블 

```python
from sklearn.model_selection import KFold
from xgboost import XGBRegressor
from lightgbm import LGBMRegressor
from sklearn.metrics import root_mean_squared_log_error

models_xgb = []
models_lgb = []
rmsle_scores = []

# 시드 리스트 설정
seeds = [0, 26, 100]

# 각 시드에 대해 반복
for index, seed in enumerate(seeds):
    print(f"===== {index+1}회차 시드: {seed} =====")
    
    # Stratified K-Fold 설정
    cv = KFold(n_splits=5, shuffle=True, random_state=seed)
    
    # K-Fold 교차 검증
    for fold_idx, (train_index, valid_index) in enumerate(cv.split(x_train, y_train), 1):
        # 학습 및 검증 데이터 분할
        X_train, X_valid = x_train.iloc[train_index], x_train.iloc[valid_index]
        Y_train, Y_valid = y_train[train_index], y_train[valid_index]

        # 모델 초기화 및 학습
        model_xgb = XGBRegressor(n_estimators=10, random_state=seed)
        model_xgb.fit(X_train, Y_train)

        model_lgb = LGBMRegressor(n_estimators=10, random_state=seed)
        model_lgb.fit(X_train, Y_train)

        # 예측 및 점수 계산
        pred_xgb = model_xgb.predict(X_valid)
        pred_lgb = model_lgb.predict(X_valid)
        fold_pred = (pred_xgb + pred_lgb) / 2
        score_ensemble = root_mean_squared_log_error(Y_valid, fold_pred)
        
        # 모델 및 점수 저장
        models_xgb.append(model_xgb)
        models_lgb.append(model_lgb)
        rmsle_scores.append(score_ensemble)

        # 폴드별 점수 출력
        print(f"폴드 {fold_idx} - RMSLE: {round(score_ensemble, 4)}")

    print("-" * 40)

# 전체 교차 검증의 평균 RMSLE 출력
print(f"검증평균 RMSLE: {round(np.mean(rmsle_scores), 4)}")
```

## 시드 기반 KFold 앙상블 예측 

```python
# 첫 번째
preds_1 = []
for idx in range(len(models_xgb)):
    pred_xgb = models_xgb[idx].predict(test)
    preds_1.append(pred_xgb)

preds_1_mean = np.mean(preds_1, axis=0)


# 두 번째
preds_2 = []
for idx in range(len(models_lgb)):
    pred_lgb = models_lgb[idx].predict(test)
    preds_2.append(pred_lgb)

preds_2_mean = np.mean(preds_2, axis=0)


# 예측 결과를 submission 데이터프레임에 저장
seed_fold_submission = submission.copy()
seed_fold_submission['ECLO'] = preds_1_mean * 0.7 + preds_2_mean * 0.3

seed_fold_submission.head()
```